In [ ]:
import os
import numpy as np
import h5py
from tqdm import tqdm
from google.colab import drive

# ---------- Mount Drive (idempotent) ----------
drive.mount("/content/drive", force_remount=False)

BASE_FOLDER = "/content/drive/My Drive/V1_Drift/repDrift_expand"
PRF_FOLDER  = "/content/drive/My Drive/V1_Drift/prfsample_expand"

def _zstr(to_zscore: int) -> str:
    return ['', '_zscore', '_zeroMean', '_equalStd', '_zeroROImean'][to_zscore]

def _has_required_files(isub: int, vreg: int, to_zscore: int) -> bool:
    """Check both the combined-ROI NPZ and the pRF H5 exist for this subject/region."""
    z = _zstr(to_zscore)
    sess_path = os.path.join(BASE_FOLDER, f"v{vreg}", f"Subject {isub}",
                             f"regressSessCombineROI_sub{isub}{z}.npz")
    prf_path  = os.path.join(PRF_FOLDER,  f"prfSampleStim_v{vreg}_sub{isub}.h5")
    return os.path.exists(sess_path) and os.path.exists(prf_path)

def discover_subjects(vreg: int, to_zscore: int, pool=range(1,9)):
    """Return the list of subjects in 1..8 that have all inputs present."""
    found = []
    for s in pool:
        if _has_required_files(s, vreg, to_zscore):
            found.append(s)
    return found

def sim_pop_response_expand(isub: int, vreg: int = 1, to_zscore: int = 0, nimgs: int = 100):
    """
    Simulate population responses by combining per-session regression
    coefficients with pRF-sampled feature maps (levels & orientations).
    Mirrors MATLAB simPopResponse_expand.m for a single subject/region.
    """
    z = _zstr(to_zscore)

    # --- Load combined-ROI regression results (NPZ) ---
    sess_path = os.path.join(BASE_FOLDER, f"v{vreg}", f"Subject {isub}",
                             f"regressSessCombineROI_sub{isub}{z}.npz")
    D = np.load(sess_path, allow_pickle=True)
    vox_coef     = D[f"v{vreg}_vox_coef"]      # [nsess, nvox, L+3] -> typically 10
    vox_ori_coef = D[f"v{vreg}_vox_ori_coef"]  # [nsess, nvox, L*O+3] -> typically 59
    nsess, nvox  = vox_coef.shape[:2]

    # --- Load pRF samples (HDF5) ---
    prf_path = os.path.join(PRF_FOLDER, f"prfSampleStim_v{vreg}_sub{isub}.h5")
    with h5py.File(prf_path, "r") as f:
        num_levels       = int(f.attrs["numLevels"])
        num_orientations = int(f.attrs["numOrientations"])

        if vreg < 4:
            lev  = np.concatenate([f["prfSampleLev/roi_0"][:],
                                   f["prfSampleLev/roi_1"][:]], axis=1)  # [img, vox, L+2]
            levO = np.concatenate([f["prfSampleLevOri/roi_0"][:],
                                   f["prfSampleLevOri/roi_1"][:]], axis=1)  # [img, vox, L+2, O]
        else:
            lev  = f["prfSampleLev/roi_0"][:]     # [img, vox, L+2]
            levO = f["prfSampleLevOri/roi_0"][:]  # [img, vox, L+2, O]

    # --- Clip nimgs to available images ---
    nimgs_avail = lev.shape[0]
    if nimgs > nimgs_avail:
        nimgs = nimgs_avail
    sim_imgs = np.arange(1, nimgs + 1, dtype=int)  # MATLAB-like 1-based selection

    # --- Allocate outputs (nvox × nsess × nimgs) ---
    vox_sess_resp     = np.zeros((nvox, nsess, nimgs), dtype=np.float32)
    vox_sess_resp_ori = np.zeros((nvox, nsess, nimgs), dtype=np.float32)

    # --- Simulate responses voxel by voxel ---
    for ivox in tqdm(range(nvox), desc=f"S{isub} V{vreg}: sim responses", ncols=100):
        # Level features (+ two SF extras + intercept) → width = num_levels + 2 + 1
        X_lev_core = lev[sim_imgs - 1, ivox, :num_levels]                   # [nimgs, L]
        X_extras   = lev[sim_imgs - 1, ivox, num_levels:num_levels+2]       # [nimgs, 2]
        X_lev = np.concatenate([X_lev_core, X_extras, np.ones((nimgs, 1))], axis=1)

        # Orientation features: take first L levels, flatten in MATLAB (Fortran) order,
        # append the same two SF extras and an intercept → width = L*O + 2 + 1
        X_ori_3d = levO[sim_imgs - 1, ivox, :num_levels, :]                 # [nimgs, L, O]
        X_ori = X_ori_3d.reshape(nimgs, num_levels * num_orientations)
        X_ori    = np.concatenate([X_ori, X_extras, np.ones((nimgs, 1))], axis=1)

        # Session-wise multiplication with coefficients
        for isess in range(nsess):
            vox_sess_resp[ivox, isess, :]     = X_lev @ vox_coef[isess, ivox, :]
            vox_sess_resp_ori[ivox, isess, :] = X_ori @ vox_ori_coef[isess, ivox, :]

    # --- Save NPZ next to regression files for this subject/region ---
    out_path = os.path.join(BASE_FOLDER, f"v{vreg}", f"Subject {isub}",
                            f"simPopResp_v{vreg}_sub{isub}{z}.npz")
    np.savez_compressed(out_path,
                        voxSessResp=vox_sess_resp,
                        voxSessRespOri=vox_sess_resp_ori,
                        simImgs=sim_imgs,
                        nsessions=nsess)
    print(f"✓ Saved NPZ: {out_path}")
    print(f"Shapes → voxSessResp: {vox_sess_resp.shape} | voxSessRespOri: {vox_sess_resp_ori.shape}")

def run_sim_for_subjects(subjects=None, vreg: int = 1, to_zscore: int = 0, nimgs: int = 100):
    """
    MATLAB-like driver: loops subjects 1..8 (or a provided list) and runs simulation
    for those that have the required inputs.
    """
    if subjects is None:
        subjects = discover_subjects(vreg=vreg, to_zscore=to_zscore, pool=range(1,9))
        print(f"Discovered ready subjects (v{vreg}, z={to_zscore}): {subjects}")
    else:
        # filter to those that actually have the files
        subjects = [s for s in subjects if _has_required_files(s, vreg, to_zscore)]
        print(f"Using provided subjects with files present: {subjects}")

    if not subjects:
        print("No subjects found with required inputs — nothing to do.")
        return

    for s in subjects:
        try:
            sim_pop_response_expand(isub=s, vreg=vreg, to_zscore=to_zscore, nimgs=nimgs)
        except Exception as e:
            print(f"[WARN] Subject {s} failed: {e}")

# ---------------- Examples ----------------
# 1) Exact MATLAB-style (runs everyone with files present in 1..8):
run_sim_for_subjects(vreg=1, to_zscore=0, nimgs=100)

# 2) Run a subset explicitly (will still skip if files missing):
# run_sim_for_subjects(subjects=[1,2,3,4,5,6,7,8], vreg=1, to_zscore=0, nimgs=100)

# 3) Single subject:
# sim_pop_response_expand(isub=3, vreg=1, to_zscore=0, nimgs=100)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Discovered ready subjects (v1, z=0): [3]


S3 V1: sim responses: 100%|███████████████████████████████████| 1254/1254 [00:00<00:00, 3403.75it/s]


✓ Saved NPZ: /content/drive/My Drive/V1_Drift/repDrift_expand/v1/Subject 3/simPopResp_v1_sub3.npz
Shapes → voxSessResp: (1254, 31, 100) | voxSessRespOri: (1254, 31, 100)


#check output

In [ ]:
import numpy as np, os

# ---------- CONFIG ----------
isub, vreg, to_zscore = 3, 1, 0
zstr = ['', '_zscore', '_zeroMean', '_equalStd', '_zeroROImean'][to_zscore]
base = "/content/drive/My Drive/V1_Drift/repDrift_expand"
path = os.path.join(base, f"v{vreg}", f"Subject {isub}", f"simPopResp_v{vreg}_sub{isub}{zstr}.npz")

# ---------- LOAD ----------
if not os.path.exists(path):
    raise FileNotFoundError(f"File not found: {path}")

d = np.load(path)
print(f"Loaded: {path}")

# ---------- BASIC INFO ----------
for k in d.files:
    v = d[k]
    print(f"{k:15s} shape={v.shape}  dtype={v.dtype}  nan%={np.isnan(v).mean()*100 if np.issubdtype(v.dtype, np.floating) else 0:.2f}%")

vox_sess_resp = d["voxSessResp"]
vox_sess_resp_ori = d["voxSessRespOri"]
nsess = d["nsessions"]
nvox, nsess, nimgs = vox_sess_resp.shape

print("\n--- Shape sanity ---")
print(f"voxSessResp:     (nvox={nvox}, nsess={nsess}, nimgs={nimgs})")
print(f"voxSessRespOri:  (nvox={vox_sess_resp_ori.shape[0]}, nsess={vox_sess_resp_ori.shape[1]}, nimgs={vox_sess_resp_ori.shape[2]})")

# ---------- VALUE CHECKS ----------
print("\n--- Basic range checks ---")
print(f"voxSessResp     min={vox_sess_resp.min():.4f}  max={vox_sess_resp.max():.4f}  mean={vox_sess_resp.mean():.4f}")
print(f"voxSessRespOri  min={vox_sess_resp_ori.min():.4f}  max={vox_sess_resp_ori.max():.4f}  mean={vox_sess_resp_ori.mean():.4f}")

# ---------- CROSS-VERIFY ONE RANDOM ENTRY ----------
ivox, isess, iimg = 0, 0, 0
pred_val = vox_sess_resp[ivox, isess, iimg]
print(f"\nExample voxel/session/image: v={ivox}, s={isess}, img={iimg} → value={pred_val:.6f}")

# ---------- HIGH-LEVEL SANITY SUMMARY ----------
print("\n✓ File structure and shapes match expectations.")
if not np.all(np.isfinite(vox_sess_resp)):
    print("⚠ Some NaN/inf values in voxSessResp.")
if not np.all(np.isfinite(vox_sess_resp_ori)):
    print("⚠ Some NaN/inf values in voxSessRespOri.")
else:
    print("✓ All values finite.")

    import numpy as np

# ratio of typical magnitude between ori vs non-ori
ratio = abs(d["voxSessRespOri"]).mean() / abs(d["voxSessResp"]).mean()
print(f"Mean magnitude ratio (ori/non-ori) = {ratio:.2e}")



Loaded: /content/drive/My Drive/V1_Drift/repDrift_expand/v1/Subject 3/simPopResp_v1_sub3.npz
voxSessResp     shape=(1254, 31, 100)  dtype=float32  nan%=0.00%
voxSessRespOri  shape=(1254, 31, 100)  dtype=float32  nan%=0.00%
simImgs         shape=(100,)  dtype=int64  nan%=0.00%
nsessions       shape=()  dtype=int64  nan%=0.00%

--- Shape sanity ---
voxSessResp:     (nvox=1254, nsess=31, nimgs=100)
voxSessRespOri:  (nvox=1254, nsess=31, nimgs=100)

--- Basic range checks ---
voxSessResp     min=-33.2192  max=46.6792  mean=3.2247
voxSessRespOri  min=-81.8577  max=66.7812  mean=3.2470

Example voxel/session/image: v=0, s=0, img=0 → value=2.285533

✓ File structure and shapes match expectations.
✓ All values finite.
Mean magnitude ratio (ori/non-ori) = 1.04e+00
